# HMM Price And Microstructure Factor Models

Notebook fits two separate Gaussian HMM grids from the compact unified dataset: one on price factors and one on microstructure factors. Each grid uses diagonal covariance, seed 42, 500 EM iterations, and `n_components` from 2 through 6.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

load_dotenv(Path.cwd() / '.env')

MODEL_SPECS = {
    'price': {
        'description': 'Price factor HMM',
        'features': [
            'ada_cum_return_30m',
            'ada_realized_volatility_20m',
            'ada_kaufman_efficiency_90m',
            'btc_log_return_30m',
        ],
        'result_prefix': 'analysis/hmm_price_factor_models_v1',
    },
    'microstructure': {
        'description': 'Microstructure factor HMM',
        'features': [
            'ada_aggression_delta_norm_10m',
            'ada_volume_entropy_norm_5m',
            'ada_pressure_concentration_30m',
            'ada_average_trade_size_quote_30m',
        ],
        'result_prefix': 'analysis/hmm_microstructure_factor_models_v1',
    },
}

N_COMPONENTS_GRID = [2, 3, 4, 5, 6]
COVARIANCE_TYPE = 'diag'
RANDOM_STATE = 42
N_ITER = 500
TOL = 1e-3
MIN_COVAR = 1e-3

TEST_CUTOFF = pd.Timestamp('2025-02-01T00:00:00Z')
TRAIN_START = TEST_CUTOFF - pd.DateOffset(years=1)
DEGENERATE_STATE_SHARE_THRESHOLD = 0.01

BUCKET = os.getenv('YC_BUCKET', 'binance-data-downloader')
SOURCE_KEY = 'features/compact/unified_dataset_full.parquet'

required_env = ['YC_ENDPOINT', 'YC_REGION', 'YC_ACCESS_KEY_ID', 'YC_SECRET_ACCESS_KEY']
missing = [name for name in required_env if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing S3 environment variables: {missing}')

s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('YC_ENDPOINT'),
    region_name=os.getenv('YC_REGION'),
    aws_access_key_id=os.getenv('YC_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('YC_SECRET_ACCESS_KEY'),
)

all_features = sorted({feature for spec in MODEL_SPECS.values() for feature in spec['features']})

print(f'Train window: {TRAIN_START} <= timestamp < {TEST_CUTOFF}')
print(f'Input: s3://{BUCKET}/{SOURCE_KEY}')
for name, spec in MODEL_SPECS.items():
    print(f'{name}: s3://{BUCKET}/{spec["result_prefix"]}/')


In [ ]:
def hmm_parameter_count(n_components: int, n_features: int, covariance_type: str) -> int:
    startprob_params = n_components - 1
    transmat_params = n_components * (n_components - 1)
    mean_params = n_components * n_features
    if covariance_type == 'diag':
        covariance_params = n_components * n_features
    elif covariance_type == 'full':
        covariance_params = n_components * n_features * (n_features + 1) // 2
    else:
        raise ValueError(f'Unsupported covariance_type: {covariance_type}')
    return startprob_params + transmat_params + mean_params + covariance_params


def sequence_run_metrics(states: np.ndarray, n_components: int) -> tuple[float, dict[str, float]]:
    if len(states) == 0:
        return math.nan, {}
    change_positions = np.flatnonzero(np.diff(states) != 0) + 1
    starts = np.r_[0, change_positions]
    ends = np.r_[change_positions, len(states)]
    run_states = states[starts]
    run_lengths = ends - starts
    metrics = {}
    for state in range(n_components):
        state_lengths = run_lengths[run_states == state]
        metrics[f'state_{state}_mean_duration'] = float(state_lengths.mean()) if len(state_lengths) else 0.0
        metrics[f'state_{state}_run_count'] = int(len(state_lengths))
    return float(run_lengths.mean()), metrics


def theoretical_mean_duration(transmat: np.ndarray, shares: np.ndarray) -> float:
    diagonal = np.clip(np.diag(transmat), 0.0, 0.999999)
    durations = 1.0 / (1.0 - diagonal)
    return float(np.sum(shares * durations))


def upload_file(local_path: Path, key: str, content_type: str | None = None) -> None:
    extra_args = {}
    if content_type:
        extra_args['ContentType'] = content_type
    s3.upload_file(str(local_path), BUCKET, key, ExtraArgs=extra_args or None)
    head = s3.head_object(Bucket=BUCKET, Key=key)
    print(f'Uploaded s3://{BUCKET}/{key} ({head["ContentLength"] / 1024**2:.2f} MiB)')


In [ ]:
workdir = Path(tempfile.mkdtemp(prefix='hmm_factor_models_'))
source_path = workdir / 'unified_dataset_full.parquet'
print(f'Downloading to {source_path}')
s3.download_file(BUCKET, SOURCE_KEY, str(source_path))

columns = ['timestamp', *all_features]
raw_data = pd.read_parquet(source_path, columns=columns)
raw_data['timestamp'] = pd.to_datetime(raw_data['timestamp'], utc=True)
raw_data = raw_data.loc[raw_data['timestamp'].ge(TRAIN_START) & raw_data['timestamp'].lt(TEST_CUTOFF)].copy()
raw_data = raw_data.sort_values('timestamp').reset_index(drop=True)

for feature in all_features:
    raw_data[feature] = pd.to_numeric(raw_data[feature], errors='raise').astype('float64')

print(f'Raw rows: {len(raw_data):,}')
print(f'Timestamp range: {raw_data["timestamp"].min()} -> {raw_data["timestamp"].max()}')
raw_data[all_features].isna().sum().sort_values(ascending=False)


In [ ]:
def fit_model_spec(spec_name: str, spec: dict) -> dict:
    features = spec['features']
    result_prefix = spec['result_prefix']
    data = raw_data[['timestamp', *features]].dropna(subset=features).reset_index(drop=True)
    values = data[features].to_numpy(dtype='float64')
    if not np.isfinite(values).all():
        raise ValueError(f'{spec_name}: training window contains non-finite feature values')

    scaler = StandardScaler()
    X = scaler.fit_transform(values).astype('float64')
    n_observations, n_features = X.shape

    print(f'\n=== {spec_name}: {spec["description"]} ===')
    print(f'Rows after dropna: {n_observations:,}; features: {n_features}')

    records = []
    best_model = None
    best_record = None
    started_at = datetime.now(timezone.utc)

    for idx, n_components in enumerate(N_COMPONENTS_GRID, start=1):
        label = f'[{idx}/{len(N_COMPONENTS_GRID)}] {spec_name} n={n_components} cov={COVARIANCE_TYPE} seed={RANDOM_STATE}'
        print(label, flush=True)
        model = GaussianHMM(
            n_components=n_components,
            covariance_type=COVARIANCE_TYPE,
            random_state=RANDOM_STATE,
            n_iter=N_ITER,
            tol=TOL,
            min_covar=MIN_COVAR,
        )
        record = {
            'spec_name': spec_name,
            'n_components': n_components,
            'covariance_type': COVARIANCE_TYPE,
            'random_state': RANDOM_STATE,
            'n_iter': N_ITER,
            'tol': TOL,
            'min_covar': MIN_COVAR,
            'n_observations': n_observations,
            'n_features': n_features,
            'status': 'ok',
        }
        try:
            model.fit(X)
            log_likelihood = float(model.score(X))
            n_parameters = hmm_parameter_count(n_components, n_features, COVARIANCE_TYPE)
            aic = float(2 * n_parameters - 2 * log_likelihood)
            bic = float(math.log(n_observations) * n_parameters - 2 * log_likelihood)
            states = model.predict(X)
            counts = np.bincount(states, minlength=n_components)
            shares = counts / counts.sum()
            avg_duration, duration_metrics = sequence_run_metrics(states, n_components)

            record.update(
                {
                    'log_likelihood': log_likelihood,
                    'aic': aic,
                    'bic': bic,
                    'n_parameters': n_parameters,
                    'converged': bool(model.monitor_.converged),
                    'em_iterations': int(model.monitor_.iter),
                    'average_sequence_duration': avg_duration,
                    'average_theoretical_duration': theoretical_mean_duration(model.transmat_, shares),
                    'min_state_share': float(shares.min()),
                    'has_degenerate_state': bool(shares.min() < DEGENERATE_STATE_SHARE_THRESHOLD),
                }
            )
            for state in range(n_components):
                record[f'state_{state}_share'] = float(shares[state])
                record[f'state_{state}_count'] = int(counts[state])
                record[f'state_{state}_self_transition'] = float(model.transmat_[state, state])
            record.update(duration_metrics)

            if best_record is None or bic < best_record['bic']:
                best_record = dict(record)
                best_model = model
        except Exception as exc:
            record.update({'status': 'failed', 'error': repr(exc)})
            print(f'  failed: {exc}', flush=True)
        finally:
            records.append(record)
            if best_model is not model:
                del model
            gc.collect()

    finished_at = datetime.now(timezone.utc)
    metrics = pd.DataFrame(records).sort_values(['status', 'bic'], na_position='last').reset_index(drop=True)
    if best_model is None or best_record is None:
        raise RuntimeError(f'{spec_name}: no HMM model finished successfully')

    best_states = best_model.predict(X)
    state_profiles = []
    for state in range(best_model.n_components):
        mask = best_states == state
        row = {
            'state': state,
            'count': int(mask.sum()),
            'share': float(mask.mean()),
            'self_transition': float(best_model.transmat_[state, state]),
        }
        for feature in features:
            row[f'{feature}_mean'] = float(data.loc[mask, feature].mean())
            row[f'{feature}_std'] = float(data.loc[mask, feature].std(ddof=1))
        state_profiles.append(row)

    states_frame = pd.DataFrame({'timestamp': data['timestamp'], 'state': best_states.astype('int16')})
    profiles = pd.DataFrame(state_profiles)
    near_best = metrics.loc[metrics['status'].eq('ok')].copy()
    near_best = near_best.loc[near_best['bic'] <= best_record['bic'] * 1.01]
    near_best = near_best.sort_values(
        ['has_degenerate_state', 'average_sequence_duration', 'bic'],
        ascending=[True, False, True],
    ).head(20)

    summary = {
        'spec_name': spec_name,
        'description': spec['description'],
        'started_at': started_at.isoformat(),
        'finished_at': finished_at.isoformat(),
        'bucket': BUCKET,
        'input_key': SOURCE_KEY,
        'result_prefix': result_prefix,
        'train_start': TRAIN_START.isoformat(),
        'test_cutoff': TEST_CUTOFF.isoformat(),
        'features': features,
        'grid': {
            'n_components': N_COMPONENTS_GRID,
            'covariance_type': COVARIANCE_TYPE,
            'random_state': RANDOM_STATE,
            'n_iter': N_ITER,
            'tol': TOL,
            'min_covar': MIN_COVAR,
        },
        'selection_rule': 'Primary criterion is minimum BIC. Near-BIC candidates are listed for interpretability review.',
        'degenerate_state_share_threshold': DEGENERATE_STATE_SHARE_THRESHOLD,
        'best_model': best_record,
        'near_best_candidates': near_best.to_dict(orient='records'),
    }

    run_dir = workdir / 'results' / spec_name
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = run_dir / 'grid_search_metrics.parquet'
    metrics_csv_path = run_dir / 'grid_search_metrics.csv'
    profiles_path = run_dir / 'best_model_state_profiles.parquet'
    states_path = run_dir / 'best_model_train_states.parquet'
    bundle_path = run_dir / 'best_model_bundle.joblib'
    summary_path = run_dir / 'summary.json'
    params_path = run_dir / 'run_config.json'

    metrics.to_parquet(metrics_path, index=False, compression='zstd')
    metrics.to_csv(metrics_csv_path, index=False)
    profiles.to_parquet(profiles_path, index=False, compression='zstd')
    states_frame.to_parquet(states_path, index=False, compression='zstd')
    joblib.dump({'model': best_model, 'scaler': scaler, 'features': features, 'best_record': best_record}, bundle_path)
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    params_path.write_text(json.dumps(summary['grid'], indent=2), encoding='utf-8')

    artifacts = [
        (metrics_path, 'grid_search_metrics.parquet', 'application/vnd.apache.parquet'),
        (metrics_csv_path, 'grid_search_metrics.csv', 'text/csv'),
        (profiles_path, 'best_model_state_profiles.parquet', 'application/vnd.apache.parquet'),
        (states_path, 'best_model_train_states.parquet', 'application/vnd.apache.parquet'),
        (bundle_path, 'best_model_bundle.joblib', 'application/octet-stream'),
        (summary_path, 'summary.json', 'application/json'),
        (params_path, 'run_config.json', 'application/json'),
    ]

    manifest = []
    for local_path, name, content_type in artifacts:
        key = f'{result_prefix}/{name}'
        upload_file(local_path, key, content_type)
        manifest.append({'name': name, 's3_key': key, 'size_bytes': local_path.stat().st_size})

    manifest_path = run_dir / 'manifest.json'
    manifest_path.write_text(json.dumps({'artifacts': manifest}, indent=2), encoding='utf-8')
    upload_file(manifest_path, f'{result_prefix}/manifest.json', 'application/json')

    print(json.dumps(best_record, indent=2))
    print(f'Done: s3://{BUCKET}/{result_prefix}/')
    return {'metrics': metrics, 'profiles': profiles, 'states': states_frame, 'summary': summary}


In [ ]:
results = {}
for spec_name, spec in MODEL_SPECS.items():
    results[spec_name] = fit_model_spec(spec_name, spec)

pd.DataFrame([result['summary']['best_model'] for result in results.values()])
